In [0]:
from pyspark.sql import functions as F

In [0]:
catalog = 'finrisk360.default.'
df = spark.read.table(catalog+'brz_bank_transactions')
df.show(5)

In [0]:
drop_features = ['transaction_id', 'customer_id','transaction_date',
                 'transaction_time','account_balance','transaction_status',
                 'has_loan']

df = df.withColumn('transaction_minute', F.minute(F.col("transaction_time"))) \
       .withColumn('loan_type', F.when((F.col('has_loan') == 1) & (F.col('loan_type') == 'None'),
                                        F.lit('Unknown'))
                                    .otherwise(F.col('loan_type')))

df = df.drop(*drop_features)
df.show(5)

In [0]:
from src.feature_engineering import EncodingPipeline

onhot_encode_features = ["transaction_direction", "account_type", "channel", "kyc_status"]
freq_encode_features = ["transaction_type", "merchant_category", "state", "loan_type"]

encoder = EncodingPipeline(one_hot=onhot_encode_features, freq=freq_encode_features)


In [0]:
df = df.repartition("transaction_hour")
df = encoder.transform(df)
df.show(5)